<!--
  id: mcp-telecom-demo
  type: example
  day: 1_6
  competency: 1.4.2
  use_case: telecom
-->

# Example 1.6 — Connecting Tools with MCP

A telecom support agent needs to check network outages, but that data lives in a system your team doesn't own the code for. This notebook stands up a tiny MCP server for it and connects an agent to it — no custom integration code.

Companion to [Problem Solution Ladder: MCP](problem-solution-ladder-1.4.2.qmd).

## Setup

In [ ]:
%pip install -q -U langchain langchain-google-genai langgraph langchain-mcp-adapters fastmcp

In [ ]:
import logging
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

# MCP tool schemas carry a JSON Schema key Gemini ignores — quiet the repeated warning
logging.getLogger("langchain_google_genai._function_utils").setLevel(logging.ERROR)

print("API key loaded!")

## Step 1 — Write the MCP server

This is the "someone else's system" — imagine the network team maintains this file, not you. It gets written to disk, then started as its own process that your agent talks to over HTTP.

In [ ]:
%%writefile network_server.py
from fastmcp import FastMCP

mcp = FastMCP("Network")

@mcp.tool()
def check_outage(zip_code: str) -> str:
    """Check whether there is a known network outage in a ZIP code."""
    known_outages = {"84604": "Outage reported, ETA 2 hours", "84601": "No known outages"}
    return known_outages.get(zip_code, "No data for that ZIP code")

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000)

In [ ]:
import subprocess, sys, time, urllib.request

server = subprocess.Popen([sys.executable, "network_server.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/mcp", timeout=1)
        break
    except urllib.error.HTTPError:
        break          # server answered (405/406) — it's up
    except Exception:
        time.sleep(1)

print("MCP server running on http://127.0.0.1:8000/mcp")

## Step 2 — Connect an agent to it

Notice this cell never imports anything from `network_server.py` — it only knows a URL. `client.get_tools()` is the entire integration.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient({
    "network": {
        "transport": "http",
        "url": "http://127.0.0.1:8000/mcp",
    }
})

tools = await client.get_tools()
print([t.name for t in tools])  # generated straight from the server, no hand-written wrapper

agent = create_agent("google_genai:gemini-2.5-flash", tools)

## Step 3 — Try it

In [ ]:
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Is there an outage near 84604?"}]}
)
print(response["messages"][-1].content)

response2 = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What about 90210?"}]}
)
print(response2["messages"][-1].content)

## Try it yourself

Add a second `@mcp.tool()` to `network_server.py`, then rerun the `%%writefile` cell, restart the server (`server.terminate()` first, then rerun the launch cell), and rerun Step 2 and Step 3. You should not need to change anything about how the agent is wired up.